In [ ]:
import sys; sys.path.extend(['..', '../Stretch2Relax'])
import MeshFEM, mesh, mesh_energy, param_utils, benchmark, viewer
import numpy as np
import sim_utils, param_utils, initial_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import fast_newton_flow, newton_flow_utils, py_newton_optimizer
from Stretch2Relax import extra_utils

# Load Meshes and compute Tutte embedding

In [ ]:
m = param_utils.load('../../models/hilbert_curve.msh.xz')
m = param_utils.load('../../models/hilbert_curve_small.msh.xz')
# m = param_utils.load('../../models/lucy.msh.xz')
# m = param_utils.load('../../models/cow2Disc.msh')
# m = param_utils.load('../../models/bird_small.msh.xz')
tutte_uv = param_utils.tutteInitialization(m)
v = mesh_energy.NodalVars(m, 2)
v.setVars(tutte_uv.ravel())
m_2d = mesh.Mesh(np.zeros_like(tutte_uv), m.elements())
m_2d.reembedElements(m.vertices())

In [ ]:
fnf = fast_newton_flow.symmetric_dirichlet(m_2d, v)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [fnf])

In [ ]:
scale = initial_utils.initialization_scale(m, v, fnf, 'grad_minimal')
prob.setVars(scale * prob.getVars())

In [ ]:
# Nullspace pinning strategy
# TODO: add epsilon * low rank.
FIX_VARS = False
if FIX_VARS:
    fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    prob.setFixedVars(fv)
else:
    # nf.elementHessianShift = 1e-8
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = False

In [ ]:
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
opt.options.niter = 200

In [ ]:
# fnf.elementHessianShift = 1e-6

In [ ]:
# fnf.elementHessianProjectionMasks = []

In [ ]:
# prob.hessianShift = 1e-6
# prob.useRelativeHessianShift = False

In [ ]:
# benchmark.reset()
# opt.optimize()
# benchmark.report()

In [ ]:
# opt.options.hessianProjectionController = py_newton_optimizer.MaskedHessianProjectionControllerGradNorm(fnf)

In [ ]:
# opt.options.hessianProjectionController.verbose = True

In [ ]:
import continuation

In [ ]:
# prob.setVars(np.load('../lucy_pre_newton.npz')['uv'].ravel())

### For NewtonFlow Baseline comparisons

In [ ]:
def eval_linear_extrapolate(x_0, coeffs, alphas):
    return np.array([(x_0 + coeffs[0] * a).reshape(-1, 2) for a in alphas])

In [ ]:
import newton_flow_utils as nfu
fv = nfu.ground_truth_flow(opt, 1, verbose=True, grad_tol=1e-8)

In [ ]:
# import newton_flow_utils as nfu
# fv = nfu.ground_truth_flow(opt, 0.01, verbose=True, grad_tol=1e-8)

In [ ]:
def grad_at(uv):
    prob.setVars(uv.ravel())
    return prob.gradient()
gradients = [grad_at(uv) for uv in fv]

In [ ]:
ghat_0 = gradients[0] / np.linalg.norm(gradients[0])
gnorms = [np.linalg.norm(g) for g in gradients]

In [ ]:
plt.semilogy(gnorms)

In [ ]:
cos_thetas = [g.dot(ghat_0) / np.linalg.norm(g) for g in gradients]

In [ ]:
plt.plot(np.acos(cos_thetas))

In [ ]:
plt.plot(np.acos(cos_thetas))

## Method Comparison

In [ ]:
import importlib
importlib.reload(extra_utils);
importlib.reload(visualization);

In [ ]:
stepper = visualization.FlowStepper(opt, initial_uv=fv[0])
stepper.show()